# PolicySpace Manual

PolicySpace is a game about economies and policy. It is being built one system at a time, and so far two systems exist: **the world** and **the people in it**. A *setting* describes a world: its goods, its map with a currency on every hex, its natural resources, the prices its markets post, and the groups of people, called **POPs**, who live on it. The engine loads the setting and runs turns in which the POPs earn, trade, consume and grow, and a developer tool lets you watch all of it happen. There is no player-facing game yet, so this manual is written for **setting authors** and developers.

*This manual grows as each part of the project is completed.*

## Contents

- [Project overview](#Project-overview)
- [Writing a setting](#Writing-a-setting)
  - [The folder layout](#The-folder-layout)
  - [Naming rules](#Naming-rules)
  - [Goods](#Goods)
  - [The map](#The-map), including each hex's currency
  - [Natural resources](#Natural-resources)
  - [Markets](#Markets)
  - [POP types](#POP-types), including how POPs grow
  - [POPs](#POPs)
  - [Running a setting](#Running-a-setting): the turn, how POPs trade and grow, turn reports
  - [Checking a setting](#Checking-a-setting)
  - [The developer UI](#The-developer-UI)
- [Technical documentation](#Technical-documentation)

## Project overview

The project is a Rust workspace with two parts: an **engine** library, which holds the rules and runs the world, and a **developer UI** program, which loads settings and shows them. The engine never depends on the developer UI. Details are in [docs/workspace.md](docs/workspace.md), and the way names become internal IDs is explained in [docs/ids.md](docs/ids.md).

Two commands matter day to day, both run from the project's root folder:

- `cargo run -p policyspace-dev-ui -- <folder>` opens the developer UI on a setting (see [The developer UI](#The-developer-UI)).
- `python tools/check.py` runs formatting checks, lint checks and all the tests.

## Writing a setting

A **setting** is a folder of plain-text [TOML](https://toml.io) files that describes a world: what goods exist, what the map looks like and which good each hex uses as its currency, where natural resources lie, what the markets charge, and which groups of people live where. The engine has no built-in content, so everything comes from the setting. A complete example lives in `settings/sample/`; copy it to start your own.

For the technical side of the format, see [docs/setting-loader.md](docs/setting-loader.md).

### The folder layout

```
my_setting/
  setting.toml      the manifest
  rules/            things that never change during play (goods, resource types, POP types)
  start/            what turn 0 looks like (the map and its currencies, the deposits, the markets, the POPs)
```

**The manifest, `setting.toml`,** has these keys:

| Key | Required | Meaning |
|---|---|---|
| `name` | yes | The setting's name |
| `description` | no | A sentence about it |
| `format_version` | yes | Always `1` for now |

A minimal manifest:

```toml
name = "Sample Setting"
description = "The project's sample setting."
format_version = 1
```

Unknown keys are errors everywhere in a setting, not just in the manifest. A misspelled key is reported instead of silently ignored.

**Files and folders.**

- Organize files however you like, nested to any depth. Files are read in sorted path order.
- What a file defines comes from its section headers (such as `[[good]]`), not from its file name.
- `README.md` files are the place for notes. Hidden files, `Thumbs.db` and `desktop.ini` are ignored. Any other file that does not end in `.toml` is an error.
- Each section type belongs in one of the two folders: `[[good]]`, `[[resource]]` and `[[pop_type]]` in `rules/`; `[[hex]]`, `[[deposit]]`, `[[market]]` and `[[pop]]` in `start/`. Putting one in the wrong folder is an error.

### Naming rules

Names of goods, resource types and POP types use lowercase letters, digits and underscores, and must start with a letter: `copper_ore` and `grain2` are fine; `Copper`, `2grain` and `copper-ore` are not. Each kind of thing has its own list of names, so a resource type or a POP type may share a name with a good.

### Goods

A **good** is anything an actor can own. The engine has no built-in goods and no built-in categories such as labor, money or capital: the setting defines them all.

Define goods with `[[good]]` sections in `rules/`, in any file at any depth, as many per file as you like. Goods from all files are merged. A setting with no goods is valid. A `[[good]]` in `start/` is an error.

| Key | Required | Meaning |
|---|---|---|
| `name` | yes | Lowercase snake_case, unique across the whole setting |
| `perishes_after` | no | A whole number, 1 or more. Leave it out and the good never perishes. |
| `movable` | no | `true` (the default) or `false`. **It has no effect yet**; movement comes in a later stage. |

`perishes_after` counts the turn a unit appears in. A unit appearing in turn 5 with `perishes_after = 1` is gone when turn 5 ends; with `perishes_after = 3` it is gone at the end of turn 7.

```toml
# Labor: gone at the end of the turn, and tied to where it is.
[[good]]
name = "unskilled_labor"
perishes_after = 1
movable = false

# Only a name is needed: durable and movable.
[[good]]
name = "copper"
```

The sample setting defines six goods:

| Name | `perishes_after` | `movable` | Role |
|---|---|---|---|
| `unskilled_labor` | 1 | false | Labor: what the sample's POPs receive every turn |
| `unimproved_copper` | 1 | false | What copper deposits yield |
| `unimproved_grain` | 1 | false | What arable land yields |
| `copper` | never | true | A durable, movable good that artisans buy |
| `grain` | 3 | true | A perishable, movable good that POPs eat |
| `silver` | never | true | The currency every sample hex uses |

More detail: [docs/goods.md](docs/goods.md).

### The map

The **map** is the set of hex tiles in the setting's starting state. For now a hex has only a position and a currency; terrain and other attributes come later. The map goes in `start/`, which describes what turn 0 looks like. A `[[hex]]` in `rules/` is an error.

Each `[[hex]]` has exactly three required keys: the whole numbers `q` and `r`, and `currency`, the name of a good (explained below). Any other key (such as `terrain`) is an error. Hexes have no names: the coordinates are the identity, and two hexes at the same coordinates is an error that names both places. Hexes can sit in any file at any depth and are merged.

```toml
[[hex]]
q = 0
r = 0
currency = "silver"

[[hex]]
q = 1
r = 0
currency = "silver"
```

**Coordinates and orientation.** The engine fixes the orientation; a setting cannot change it. Hexes are *pointy-top*. Each value of `r` is one straight east-west row; `q` increases eastward and `r` increases southward. So `(1, 0)` is the east neighbor of `(0, 0)`, and the row `r = 1` lies south of the row `r = 0`. Values may be negative, up to about plus or minus 2.1 billion.

**Shape.** There are no shape rules. Holes, islands and negative coordinates are all fine, and a setting with no hexes is valid. Neighbors, distance and wrapping around the edges do not exist yet; wrapping is planned for later as an optional per-setting value.

**Currency.** Every hex names the good that prices on that hex are quoted in: its market posts prices in it, and the POPs there are paid in it when they sell and pay with it when they buy. Any good the setting defines can be a currency. The engine has no idea of "money", so a perishable or unmovable good is allowed, and if you name labor as the currency, that is exactly what you get. Different hexes may name different goods, and there is no map-wide default: every hex writes its own.

```toml
[[hex]]
q = 0
r = 0
currency = "coin"

[[hex]]
q = 1
r = 0
currency = "shell"     # the next hex along trades in a different good
```

A hex without a `currency` key, one whose value is not text, and one naming a good that does not exist (including a name in the wrong case, such as `"Silver"`) are each an error with file, line and column. Like the other errors that refer to names, the unknown-good one is not shown while part of the setting could not be read. Because the key is required, a setting written before currencies existed will not load until every hex has one.

The sample map is a hexagon of radius 2 around `(0, 0)`: 19 hexes, every one with `currency = "silver"`. The diagram below shows it, with each hex labelled `q, r`.

More detail: [docs/map.md](docs/map.md) for the map and [docs/currency.md](docs/currency.md) for currencies.

In [ ]:
import math
from IPython.display import SVG, display

def hex_svg(labels=None, size=36, shade="#fdf3d0", color="#a40"):
    """The sample map. Hexes listed in `labels` are shaded and get the given lines of text."""
    labels = labels or {}
    cells = [(q, r) for r in range(-2, 3) for q in range(-2, 3) if abs(q + r) <= 2]
    pts = [(size * math.sqrt(3) * (q + r / 2), size * 1.5 * r) for q, r in cells]
    minx = min(p[0] for p in pts) - size; maxx = max(p[0] for p in pts) + size
    miny = min(p[1] for p in pts) - size; maxy = max(p[1] for p in pts) + size
    w, h = maxx - minx, maxy - miny + 24
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{w:.0f}" height="{h:.0f}" viewBox="{minx:.0f} {miny - 24:.0f} {w:.0f} {h:.0f}" font-family="sans-serif">']
    out.append(f'<text x="{minx + 4:.0f}" y="{miny - 8:.0f}" font-size="12">q grows east, r grows south</text>')
    for (q, r), (x, y) in zip(cells, pts):
        corners = " ".join(
            f"{x + size * math.cos(math.radians(60 * i - 30)):.1f},{y + size * math.sin(math.radians(60 * i - 30)):.1f}"
            for i in range(6))
        fill = shade if (q, r) in labels else "#e8f0fa"
        out.append(f'<polygon points="{corners}" fill="{fill}" stroke="#456" stroke-width="1.2"/>')
        out.append(f'<text x="{x:.1f}" y="{y - 2:.1f}" font-size="11" text-anchor="middle">{q}, {r}</text>')
        for i, line in enumerate(labels.get((q, r), [])):
            out.append(f'<text x="{x:.1f}" y="{y + 9 + 10 * i:.1f}" font-size="8.5" text-anchor="middle" fill="{color}">{line}</text>')
    out.append("</svg>")
    return SVG("\n".join(out))

display(hex_svg())

### Natural resources

Natural resources come in two parts: **resource types** say what kinds of resource exist and what one deposit produces, and **deposits** put them on the map. Each deposit produces its type's output every turn (see [Running a setting](#Running-a-setting)). There is no ownership yet (that is a later stage), and no stock or depletion.

**Resource types** are `[[resource]]` sections in `rules/` (in `start/` it is an error), in any file at any depth, merged across files. Three keys are required:

| Key | Meaning |
|---|---|
| `name` | Lowercase snake_case, unique among resource types. It may match a good's name. |
| `produces` | The name of a good the setting defines |
| `output` | How much of that good one deposit produces per turn: a whole number or decimal, greater than 0 |

Each type produces exactly one good, but several types may produce the same one. By convention the produced good is unmovable and perishes after one turn (like the sample's `unimproved_*` goods), but the engine does not require that.

**Deposits** are `[[deposit]]` sections in `start/` (in `rules/` it is an error). Keys:

| Key | Meaning |
|---|---|
| `resource` | The name of a resource type |
| `q`, `r` | A hex that exists on the map |
| `count` | Optional, 1 to 1,000,000, default 1 |

`count = 3` makes three separate deposits. Each will be ownable by a single owner later, so the total on a hex is `count` times `output`. A hex can hold several resource types, but each type may appear only once per hex: raise `count` instead.

```toml
# rules/resources.toml
[[resource]]
name = "copper_ore"
produces = "unimproved_copper"
output = 2.5

# start/deposits.toml
[[deposit]]
resource = "copper_ore"
q = 1
r = -1
count = 2

[[deposit]]
resource = "copper_ore"
q = -2
r = 1
```

A resource's `produces`, a deposit's `resource`, and a deposit's hex are checked after all files are read, so the target can be in any file. These reference errors are not shown if part of the setting could not be read (see [Checking a setting](#Checking-a-setting)).

**The sample setting** has two resource types:

| Name | Produces | Output per deposit |
|---|---|---|
| `copper_ore` | `unimproved_copper` | 2.5 |
| `arable_land` | `unimproved_grain` | 4 |

and six deposit entries that place 10 deposits:

| Hex `(q, r)` | Resource | Count |
|---|---|---|
| `(0, 0)` | `arable_land` | 3 |
| `(0, 0)` | `copper_ore` | 1 |
| `(-1, 0)` | `arable_land` | 2 |
| `(1, 0)` | `arable_land` | 1 |
| `(1, -1)` | `copper_ore` | 2 |
| `(-2, 1)` | `copper_ore` | 1 |

Hex `(0, 0)` holds two resource types at once. In the diagram, hexes with deposits are shaded.

In [ ]:
display(hex_svg({
    (0, 0): ["arable_land x3", "copper_ore x1"],
    (-1, 0): ["arable_land x2"],
    (1, 0): ["arable_land x1"],
    (1, -1): ["copper_ore x2"],
    (-2, 1): ["copper_ore x1"],
}))

Planned for later: stock, depletion and regeneration as optional data on resource types, so settings written now stay valid.

More detail: [docs/resources.md](docs/resources.md), with [docs/map.md](docs/map.md) for coordinates and [docs/goods.md](docs/goods.md) for goods.

### Markets

A **market** is where the POPs on a hex buy and sell. For now it is a stand-in: it posts fixed prices and buys or sells any amount at them. It holds no stock of its own and never runs out: goods sold to it vanish, goods bought from it appear from nothing, and the currency paid or received does the same. A later stage replaces it with a real market whose prices move with supply and demand. Until then, the prices you write are what everyone trades at.

Markets are `[[market]]` sections in `start/` (in `rules/` it is an error, since prices are state that a live market will change). Keys:

| Key | Meaning |
|---|---|
| `q`, `r` | A hex that exists on the map |
| `prices` | A table of `good name = price`. Each price is a number greater than 0: how many units of the hex's currency one unit of the good costs |

The rules that follow from this:

- **A good is traded on a hex only if that hex's market lists a price for it.** A good with no price there is neither bought nor sold there, however much anyone wants it. A good may exist in `rules/` long before any hex trades it.
- A hex with no `[[market]]` at all is valid and trades nothing. So is a market with `prices = {}`.
- At most one market per hex.
- Prices are in the hex's currency (see [The map](#The-map)). The currency itself always costs 1, so listing it in `prices` is an error. A good may be priced on a hex whose currency is a different good.
- Both TOML forms of the table work: inline, or a `[market.prices]` sub-table.

```toml
# Inline: grain and fish are traded here; copper, with no price, is not.
[[market]]
q = 0
r = 0
prices = { grain = 2, fish = 1.5 }

# The same thing written as a sub-table.
[[market]]
q = 1
r = 0

[market.prices]
grain = 3
fish = 1
```

How a trade works (what POPs do with it is in [Running a setting](#Running-a-setting)): selling removes the good from the seller's stockpile and adds the price times the amount in currency; buying does the reverse. A bought good, and currency received from a sale, counts as appearing in the turn of the trade, so a perishable one starts its countdown then. When goods leave a stockpile, the units that would perish soonest go first.

Errors, each with file, line and column: a missing or bad `q` or `r`; coordinates with no hex; a second market on the same hex (the message names the first); a missing `prices` or one that is not a table; a price that is not a number or not greater than 0 (`inf` included); an unknown good in `prices`; the hex's own currency listed (`'silver' is the currency at (0, 0), so it can't have a price`); and an unknown key. The unknown-good and currency checks are skipped while part of the setting could not be read.

**The sample setting** gives 18 of its 19 hexes a market, with prices in `silver`:

| Hexes | `unskilled_labor` | `grain` | `copper` | `unimproved_grain` | `unimproved_copper` |
|---|---|---|---|---|---|
| the center `(0, 0)` | 1 | 2 | 5 | 0.5 | 1 |
| every other hex except `(2, -2)` | 1 | 2 | 5 | not traded | not traded |
| `(2, -2)`, the north-east corner | no market | | | | |

So the sample has one hex that trades every good, seventeen that trade some, and one that trades nothing. The prices are placeholders: nothing is produced for sale yet, so they need not be consistent with production.

More detail: [docs/market.md](docs/market.md).

### POP types

A **POP** is a group of identical people of one kind living on one hex, and a **POP type** describes one person of that kind: what they receive each turn, what they want to consume, and how their numbers grow. POP types are rules, so they go in `rules/` as `[[pop_type]]` sections (in `start/` it is an error). Mind the two headers: `[[pop_type]]` in `rules/` describes a person, and `[[pop]]` in `start/` places a group of them (see [POPs](#POPs)).

Four keys, all required:

| Key | Meaning |
|---|---|
| `name` | Lowercase snake_case, unique among POP types |
| `endowment` | `good name = amount` pairs: what one person receives at the start of every turn. Amounts are numbers greater than 0. May be empty (`{}`), for a type with no income |
| `utility` | One entry per good a person consumes, each with two optional numbers that default to 0: `subsistence`, the amount needed before anything else, and `share`, the good's weight when spending whatever is left. Each listed good needs at least one of them above 0, and at least one good needs a `share` above 0 |
| `growth` | How a group's size responds to its prosperity: three values described under [How POPs grow](#How-POPs-grow) below |

Things worth knowing:

- **Shares are weights, not percentages.** The engine scales them to add up to 1, so shares of `1`, `4` and `5` mean 10%, 40% and 50%. You never have to make them add up yourself.
- **A pure necessity** is a good with a `subsistence` and no `share`: a person buys it up to the subsistence amount and never more.
- **Leisure needs no special setting.** A good that is endowed *and* in the utility is kept and consumed: that is leisure. An endowed good that is not in the utility is sold entirely; a wanted good that is not endowed is bought.
- **What to endow.** The intended design gives people nothing but unskilled labor, which they sell and partly keep. Any good is allowed, the currency included, so the system can be tried out before production exists.
- **A pitfall.** The engine does not compare a good's subsistence with its endowment. For a good only the person produces, such as labor, a subsistence at or above the endowment can never be reached.
- Both TOML forms work: inline tables, or sub-tables such as `[pop_type.utility.grain]`.

#### How POPs grow

Each turn, after it has traded and consumed, a POP's size changes by a rate that depends on how far above or below subsistence it was. The measure is its **subsistence index**: how many extra subsistence bundles the POP's income could have bought. It is 0 when income exactly covers subsistence, 1 when income is twice the cost of subsistence, and negative below subsistence, down to −1 for a POP that has nothing. The `growth` table has three required keys:

| Key | Meaning |
|---|---|
| `growth` | A list of one or more numbers giving the shape of growth above subsistence: with an index *u*, the rate is `c1·u + c2·u² + …`. One number gives a straight line; a negative second number makes growth peak and then fall away as the POP gets richer |
| `decline` | A number from 0 up to (not including) 1: how fast a POP with nothing shrinks each turn. Below subsistence the shrinking is in proportion, so a POP that gets half of what it needs shrinks by half of `decline` |
| `max_growth` | A number of at least 0: the most a POP can grow in one turn |

The rate always stays between −`decline` and `max_growth`, and a POP is never removed: a starving POP loses the same fraction every turn but never reaches zero. `growth = [0]` with `decline = 0` freezes a type's size, which is useful for a setting that should stay put.

Two things about "below subsistence" matter here. A POP that cannot get a good it needs at all, because its hex does not sell it and it holds none, counts as having nothing and shrinks at the full `decline`, however rich it is otherwise. And a type whose subsistence goods cost nothing where it lives (or that has no subsistence at all) neither grows nor shrinks.

```toml
# Inline form. Farmers keep some labor as leisure and spend the rest on grain.
[[pop_type]]
name = "farmer"
endowment = { labor = 10 }
utility = { labor = { subsistence = 1, share = 1 }, grain = { subsistence = 2, share = 3 } }
growth = { growth = [0.06, -0.03], decline = 0.1, max_growth = 0.05 }

# Sub-table form. Fishers need one fish and want no more than that (a pure
# necessity), spend the rest on coin, and grow in a straight line.
[[pop_type]]
name = "fisher"

[pop_type.endowment]
fish = 2
labor = 8

[pop_type.utility.fish]
subsistence = 1

[pop_type.utility.coin]
share = 0.5

[pop_type.growth]
growth = [0.03]
decline = 0
max_growth = 1
```

Errors, each with file, line and column: a missing key; a bad or duplicate name; an `endowment`, `utility` or `growth` that is not a table; an endowment amount that is not a number above 0; a utility entry that is not a table; a negative or non-finite `subsistence` or `share`; a utility good with nothing above 0; a utility with no positive share; a `growth` list that is empty or holds something that is not a finite number; a `decline` outside 0 to 1; a negative `max_growth`; an unknown good; and an unknown key anywhere. Unknown-good errors are skipped while part of the setting could not be read. Because `growth` is required, a type written without it will not load until it is added.

**The sample setting** defines two types, both receiving 10 `unskilled_labor` per person per turn:

| Type | Good | Subsistence | Share as written | Share used |
|---|---|---|---|---|
| `peasant` | `unskilled_labor` (leisure) | 2 | 0.2 | 0.2 |
| | `grain` | 2 | 0.8 | 0.8 |
| `artisan` | `unskilled_labor` (leisure) | 1 | 1 | 0.1 |
| | `grain` | 2 | 4 | 0.4 |
| | `copper` | 0 | 5 | 0.5 |

Peasants keep some labor as leisure and spend mostly on grain; artisans keep less leisure and weight copper heavily. The artisan rows show weights being scaled to shares. Both types grow with `growth = [0.06, -0.03]`, `decline = 0.1` and `max_growth = 0.05`: growth peaks at 3% a turn for a POP whose income is twice its subsistence cost, falls back to zero at three times, and a POP with nothing shrinks 10% a turn. The chart shows that rule, with the three situations the sample's POPs are in (where those numbers come from is in [Running a setting](#Running-a-setting)).

In [ ]:
def line_chart(series, xlim, ylim, xticks, yticks, xlabel, ylabel, points=(), vlines=(),
               width=680, height=340, xfmt=lambda v: f"{v:g}", yfmt=lambda v: f"{v:g}"):
    """A simple SVG line chart. `series` is a list of (name, color, [(x, y), ...])."""
    L, R, T, B = 60, 16, 36, 46
    pw, ph = width - L - R, height - T - B
    X = lambda x: L + (x - xlim[0]) / (xlim[1] - xlim[0]) * pw
    Y = lambda y: T + (ylim[1] - y) / (ylim[1] - ylim[0]) * ph
    out = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" font-family="sans-serif" font-size="11">']
    for t in xticks:
        out.append(f'<line x1="{X(t):.1f}" y1="{T}" x2="{X(t):.1f}" y2="{T + ph}" stroke="#e5e5e5"/>')
        out.append(f'<text x="{X(t):.1f}" y="{T + ph + 14}" text-anchor="middle">{xfmt(t)}</text>')
    for t in yticks:
        out.append(f'<line x1="{L}" y1="{Y(t):.1f}" x2="{L + pw}" y2="{Y(t):.1f}" stroke="#e5e5e5"/>')
        out.append(f'<text x="{L - 6}" y="{Y(t) + 4:.1f}" text-anchor="end">{yfmt(t)}</text>')
    if ylim[0] < 0 < ylim[1]:
        out.append(f'<line x1="{L}" y1="{Y(0):.1f}" x2="{L + pw}" y2="{Y(0):.1f}" stroke="#999"/>')
    if xlim[0] < 0 < xlim[1]:
        out.append(f'<line x1="{X(0):.1f}" y1="{T}" x2="{X(0):.1f}" y2="{T + ph}" stroke="#999"/>')
    out.append(f'<rect x="{L}" y="{T}" width="{pw}" height="{ph}" fill="none" stroke="#999"/>')
    for x, label in vlines:
        out.append(f'<line x1="{X(x):.1f}" y1="{T}" x2="{X(x):.1f}" y2="{T + ph}" stroke="#a40" stroke-dasharray="4 3"/>')
        out.append(f'<text x="{X(x) + 4:.1f}" y="{T + 12}" fill="#a40">{label}</text>')
    lx = L
    for name, color, pts in series:
        d = " ".join(f"{X(x):.1f},{Y(y):.1f}" for x, y in pts)
        out.append(f'<polyline points="{d}" fill="none" stroke="{color}" stroke-width="2"/>')
        out.append(f'<line x1="{lx}" y1="{T - 16}" x2="{lx + 18}" y2="{T - 16}" stroke="{color}" stroke-width="2"/>')
        out.append(f'<text x="{lx + 22}" y="{T - 12}">{name}</text>')
        lx += 34 + 6.2 * len(name)
    for x, y, label, anchor, dy in points:
        out.append(f'<circle cx="{X(x):.1f}" cy="{Y(y):.1f}" r="4" fill="#222"/>')
        dx = 7 if anchor == "start" else -7
        out.append(f'<text x="{X(x) + dx:.1f}" y="{Y(y) + dy:.1f}" text-anchor="{anchor}">{label}</text>')
    out.append(f'<text x="{L + pw / 2:.1f}" y="{height - 6}" text-anchor="middle">{xlabel}</text>')
    out.append(f'<text transform="translate(14 {T + ph / 2:.1f}) rotate(-90)" text-anchor="middle">{ylabel}</text>')
    out.append("</svg>")
    return SVG("\n".join(out))

def growth_rate(u, coefficients=(0.06, -0.03), decline=0.1, max_growth=0.05):
    """The sample types' growth rule: a polynomial above subsistence, a straight decline below, both clamped."""
    rate = sum(c * u ** (i + 1) for i, c in enumerate(coefficients)) if u >= 0 else decline * u
    return max(-decline, min(max_growth, rate))

us = [-1 + i / 100 for i in range(0, 501)]
display(line_chart(
    [("growth rate of the sample's POP types", "#1f77b4", [(u, growth_rate(u)) for u in us])],
    xlim=(-1, 4), ylim=(-0.12, 0.06), xticks=[-1, 0, 1, 2, 3, 4], yticks=[-0.1, -0.05, 0, 0.05],
    xlabel="subsistence index (extra subsistence bundles the income could buy)", ylabel="size change per turn",
    points=[(-1, -0.1, "peasant with no market: −10%", "start", -8),
            (2 / 3, growth_rate(2 / 3), "peasant on a market hex: +2.67%", "end", 16),
            (1, growth_rate(1), "artisan: +3%", "start", -6)],
    yfmt=lambda v: f"{v * 100:+.0f}%" if v else "0%"))

More detail: [docs/pops.md](docs/pops.md) for the keys and every error message, [docs/growth.md](docs/growth.md) for the growth rule, and [docs/goods.md](docs/goods.md) for the goods a type names.

### POPs

A `[[pop]]` section in `start/` places a group of people of one type on one hex (in `rules/` it is an error, since a POP changes during play). Four keys, all required:

| Key | Meaning |
|---|---|
| `type` | The name of a POP type from `rules/` |
| `q`, `r` | A hex that exists on the map, as for deposits and markets |
| `size` | The number of people at turn 0: a number greater than 0, decimals allowed |

- **Size is a starting value, and it need not be whole.** A POP is an aggregate: what it receives and wants scales with its size, and every turn the size is multiplied by a growth rate, so after the first turn sizes are long decimals anyway. Nothing is rounded, in the engine or on screen.
- **At most one POP of each type per hex.** Two groups of the same type on one hex would behave as one group split in two, so a second `[[pop]]` with the same type and coordinates is an error that names the first. A hex may hold POPs of several types, or none.
- **POPs start with nothing.** There is no starting cash or goods. At the start of every turn, turn 0 included, each POP receives its type's endowment times its size, and it perishes like any other good: the sample's labor would be gone at the end of the turn it arrives if the POP did not sell or consume it, while an endowed durable good would pile up.
- Deposits are still unowned: their output goes to the hex's unowned stockpile, not to the POPs there.

```toml
[[pop]]
type = "peasant"
q = 0
r = 0
size = 60

[[pop]]
type = "artisan"     # a second type on the same hex is fine
q = 0
r = 0
size = 2.5           # sizes need not be whole numbers
```

Errors, each with file, line and column: a missing `type`, `q`, `r` or `size`; a `type` that is not text; bad coordinates or coordinates with no hex; a `size` that is not a number above 0; a duplicate POP; an unknown POP type; an unknown key; and a `[[pop]]` in `rules/`. The unknown-type and no-hex errors are skipped while part of the setting could not be read.

**The sample setting** places 16 POPs, 410 people in all: a `peasant` POP on 15 of the 19 hexes, with sizes from 12 to 60, and an `artisan` POP of 10 at the center `(0, 0)`, the only hex that prices every good, so the center holds two POPs. The peasant POP of 12 on the north-east corner `(2, -2)` has no market, so it can never sell its labor or buy grain; it is there to show what happens to a POP that cannot trade. Four hexes have no POP: `(2, -1)`, `(-2, 0)`, `(1, 1)` and `(-2, 2)`. In the diagram, hexes with POPs are shaded.

In [ ]:
display(hex_svg({
    (0, -2): ["peasant 20"], (1, -2): ["peasant 15"], (2, -2): ["peasant 12", "(no market)"],
    (-1, -1): ["peasant 25"], (0, -1): ["peasant 30"], (1, -1): ["peasant 35"],
    (-1, 0): ["peasant 40"], (0, 0): ["peasant 60", "artisan 10"], (1, 0): ["peasant 35"], (2, 0): ["peasant 18"],
    (-2, 1): ["peasant 22"], (-1, 1): ["peasant 28"], (0, 1): ["peasant 24"],
    (-1, 2): ["peasant 16"], (0, 2): ["peasant 20"],
}, shade="#e6f4e6", color="#264"))

More detail: [docs/pops.md](docs/pops.md).

### Running a setting

A loaded setting becomes a running **world**. The rules stay fixed; the world holds the clock (starting at turn 0), a stockpile of unowned goods on every hex, and for every POP its current size and its own stockpile. Each turn has three parts:

1. **Emit and endow.** At the start of the turn every deposit produces its resource type's `output` of its good, into the unowned stockpile of its hex (deposits have no owners yet). Then every POP receives its type's endowment times its size, into its own stockpile.
2. **Trade, consume and grow.** On every hex, each POP sells the market what it does not mean to keep, buys what it wants with the proceeds, and consumes. Then every POP's size changes by its growth rate. The rest of this section explains it.
3. **Perish.** At the end of the turn, goods that have reached their limit are removed from every stockpile, the hexes' and the POPs'.

The world is observed after part 2 and before part 3, so what you see at any turn is the world after that turn's trading and growth, with nothing perished yet. That includes turn 0: a freshly loaded world has already run turn 0's trade and growth, which is why its POPs are not at the sizes written in the setting (see [Turn reports](#Turn-reports) below).

`perishes_after` takes effect in part 3. With 1, output is usable during the turn it appears and gone when it ends. With 3, it is gone at the end of the third turn it exists. Goods without `perishes_after` last forever and accumulate. (`movable` still has no effect.)

A **stockpile** holds the total of each good, and for each total, how much perishes at the end of which turn. For example, a good with `perishes_after = 3` produced at 5 per turn:

| Turn | Held during the turn (after the emit) | Removed when the turn ends |
|---|---|---|
| 0 | 5 | 0 |
| 1 | 10 | 0 |
| 2 | 15 | 5 (the batch from turn 0) |
| 3 | 15 | 5 (the batch from turn 1) |
| 4 | 15 | 5 (the batch from turn 2) |

#### How POPs trade

Each turn, once the endowment has arrived, the market on the POP's hex works out what the POP wants to end up with and makes the trades. In plain terms:

- **Income is everything the POP holds**, valued at the hex's prices: this turn's endowment, plus any durable goods it still has, plus any currency left over from earlier turns.
- **Subsistence comes first.** The POP first covers every utility good's subsistence amount, times its size. Whatever income is left is split between its utility goods by their shares. Goods the hex does not price are left out of this sum: a held one is simply kept, and a wanted one cannot be bought.
- **Below subsistence, everything is scaled down.** If the income cannot cover the subsistence bundle, the POP spends all of it and gets the same fraction of every subsistence amount, leisure included: a peasant facing dear grain sells more of its labor than it would like to keep and still goes short of grain.
- **Sales first, then purchases.** On each hex every POP sells before any POP buys. With today's fixed-price market this changes nothing; it is there so that the later live market, which can run short, is fair.
- **Then it consumes** the goods in its utility, up to what it wanted. The labor it kept is its leisure, consumed the same way. Goods outside its utility are never consumed: they are sold if the hex prices them and kept if it does not.
- **Leftover currency stays.** A POP spends essentially all its income; floating-point rounding can leave a tiny sliver of currency, which stays in its stockpile and counts as income next turn. Nothing is rounded.
- **No market, no trade.** A POP on a hex with no market keeps what it has and consumes what it can. The sample's north-east peasant consumes all its labor as leisure and goes without grain every turn.

Some consequences for setting authors: a durable good a POP is endowed with but neither wants nor can sell piles up in its stockpile; if none of the goods a type gives a share to is priced on its hex, its spare income is kept as currency and, for a durable currency, piles up too; and a POP that cannot afford its subsistence falls short on everything in the same proportion.

A worked example from the sample at turn 0, at a typical hex's prices (labor 1, grain 2 and copper 5 silver):

| | Center peasant, size 60 | North-east peasant, size 12 (no market) |
|---|---|---|
| Receives | 600 labor | 120 labor |
| Income | 600 silver | nothing can be sold, so 0 |
| Subsistence needs | 120 labor and 120 grain, costing 360 silver | 24 labor and 24 grain, but grain cannot be bought |
| Left over | 240 silver, split 0.2 to labor and 0.8 to grain | nothing |
| Wants to end up with | 168 labor and 216 grain | the 120 labor it holds, and 0 grain |
| Sells | 432 labor for 432 silver | nothing |
| Buys | 216 grain for 432 silver | nothing |
| Consumes | 168 labor and 216 grain | 120 labor |
| Holds afterwards | at most a rounding sliver of silver | nothing |

Per person: a peasant sells 7.2 of its 10 labor, buys 3.6 grain and keeps 2.8 labor as leisure, and an artisan at the center sells 8.5 labor and buys 3 grain and 0.5 copper. The chart shows how a peasant's choices change with the price of grain. Above a price of 4 (with labor at 1) its 10 labor can no longer cover its subsistence, and everything scales down together.

In [ ]:
def peasant_choice(grain_price, labor_price=1.0, endowment=10.0):
    """What one sample peasant keeps, sells and buys at a grain price: (labor kept, labor sold, grain bought)."""
    income = endowment * labor_price
    cost = 2 * labor_price + 2 * grain_price          # subsistence: 2 labor and 2 grain
    if income >= cost:
        extra = income - cost
        kept = 2 + 0.2 * extra / labor_price
        grain = 2 + 0.8 * extra / grain_price
    else:
        fraction = income / cost
        kept, grain = 2 * fraction, 2 * fraction
    return kept, endowment - kept, grain

prices = [1 + i / 50 for i in range(0, 451)]
choices = [peasant_choice(p) for p in prices]
display(line_chart(
    [("labor sold", "#d62728", [(p, c[1]) for p, c in zip(prices, choices)]),
     ("labor kept as leisure", "#2ca02c", [(p, c[0]) for p, c in zip(prices, choices)]),
     ("grain bought", "#1f77b4", [(p, c[2]) for p, c in zip(prices, choices)])],
    xlim=(1, 10), ylim=(0, 10), xticks=list(range(1, 11)), yticks=[0, 2, 4, 6, 8, 10],
    xlabel="price of grain, in silver (labor costs 1)", ylabel="amount per peasant",
    vlines=[(4, "below subsistence from here")]))

#### How a POP's size changes

After trading and consuming, each POP's size is multiplied by one plus its growth rate (the rule and the type's parameters are under [How POPs grow](#How-POPs-grow)). In the sample a peasant on a market hex has 240 silver to spare against a subsistence cost of 360, so its index is 2/3 and its rate about +2.67% a turn; the center artisan has an index of 1 and grows 3%; and the north-east peasant, which cannot get grain at all, has an index of −1 and shrinks 10% a turn: 12 people become 10.8 at turn 0, 9.72 after one step and 8.748 after two. The rate depends only on prices and the type, not on size, so the sample's sizes compound at the same rate every turn, and the next turn's endowment, trade and consumption grow or shrink with them. The chart shows twenty turns of this, each POP relative to the size written in the setting (turn 0 already includes turn 0's growth).

In [ ]:
turns = list(range(0, 21))
display(line_chart(
    [("market-hex peasant (+2.67%)", "#2ca02c", [(t, (1 + growth_rate(2 / 3)) ** (t + 1)) for t in turns]),
     ("artisan (+3%)", "#1f77b4", [(t, 1.03 ** (t + 1)) for t in turns]),
     ("no-market peasant (−10%)", "#d62728", [(t, 0.9 ** (t + 1)) for t in turns])],
    xlim=(0, 20), ylim=(0, 2), xticks=[0, 5, 10, 15, 20], yticks=[0, 0.5, 1, 1.5, 2],
    xlabel="turn (each POP grows or shrinks by the same rate every turn)", ylabel="size, as a multiple of the setting's"))

#### Turn reports

Each turn produces a report with totals over the whole map: per good, what `perished` when the previous turn ended, what deposits `emitted`, what POPs were `endowed`, and what POPs `sold`, `bought` and `consumed`; then the total `population` after growth. Each list is in the order the goods are defined (so the sample's `unskilled_labor` comes first and `copper` before `grain`), lists only goods with an amount, and prints `nothing` when empty. The sample over turns 0 to 2, as the checker prints it (the next section gives the command):

In [ ]:
import subprocess

command = ["cargo", "run", "-q", "-p", "policyspace-dev-ui", "--", "check", "settings/sample", "--turns", "2"]
try:
    run = subprocess.run(command, capture_output=True, text=True, timeout=900)
    if run.returncode == 0:
        print(run.stdout.strip())
    else:
        print(f"The checker failed (exit code {run.returncode}):\n{run.stderr.strip()}")
except OSError as error:
    print(f"The checker could not be run: {error}")

Where the numbers come from: the setting's 410 people receive 10 labor each, 4100 in all. Each of the 388 peasants on a market hex sells 7.2 labor and buys 3.6 grain, the 10 artisans sell 8.5 labor each and buy grain and copper, and the 4100 − 2878.6 = 1221.4 labor not sold is consumed as leisure (the no-market peasant's 120 included). Labor never appears in `perished`, because it is all sold or consumed during the turn; only the deposit output perishes. The population at turn 0 is already about 419.4, not 410, because turn 0's growth has run, and each later turn's `endowed` is 10 labor times the previous turn's population. Amounts print in full, with nothing rounded, which is why they grow long tails of digits after turn 0.

More detail: [docs/world.md](docs/world.md) for the turn and the report, [docs/trade.md](docs/trade.md) for the trade round, [docs/demand.md](docs/demand.md) for how what a POP wants is worked out, and [docs/growth.md](docs/growth.md) for growth.

### Checking a setting

The checker loads a setting and reports every problem in one go, with file, line and column. It never produces a half-loaded setting.

```
dev-ui check <folder> [--turns N]
```

From the project root, that is `cargo run -p policyspace-dev-ui -- check settings/sample`. On success it prints one line:

```
Loaded setting 'Sample Setting' (format version 1): 6 goods, 19 hexes, 2 resources, 10 deposits, 18 markets, 2 pop types, 16 pops
```

(Counts use singular wording where it applies: `1 hex`, `1 resource`, `1 deposit`, `1 market`, `1 pop type`, `1 pop`. Currencies are not counted, since every hex has exactly one.) With `--turns N` (N from 0 to 10,000) it then prints one report line per turn, from turn 0 through turn N, as shown in [Turn reports](#Turn-reports); `--turns 0` prints only turn 0.

Problems look like this:

```
rules/a.toml:3:18: key 'perishes_after' in good must be at least 1
rules/a.toml:6:6: duplicate good name 'copper' (first defined at rules/a.toml:2:6)
rules/a.toml:10:10: unknown good 'gold'
rules/p.toml:14:1: utility of pop_type needs at least one good with a positive share
rules/p.toml:20:11: key 'decline' in growth of pop_type must be a finite number of at least 0 and less than 1
start/k.toml:5:3: 'coin' is the currency at (0, 0), so it can't have a price
start/m.toml:4:1: unknown key 'terrain' in hex
start/m.toml:6:1: duplicate hex at (0, 0) (first defined at start/m.toml:1:1)
start/m.toml:9:1: missing required key 'currency' in hex
start/p.toml:7:1: duplicate pop of 'peasant' at (0, 0) (first defined at start/p.toml:1:1)
10 error(s) found in my_setting
```

If part of the setting could not be read (a missing folder, or a broken or unreadable file), errors that refer to names or hexes, such as `unknown good`, are not shown; fix the structural problem first.

| Exit code | Meaning |
|---|---|
| 0 | The setting loaded |
| 1 | The setting has errors |
| 2 | Wrong usage (a three-line usage message is printed) |

The checker never opens a window, so it is the form to use in scripts. `dev-ui --version` prints the engine's name and version. More detail: [docs/setting-loader.md](docs/setting-loader.md), and [docs/world.md](docs/world.md) for `--turns`.

### The developer UI

The developer UI is a window that shows a setting's map and lets you step through turns and inspect the world: each hex's goods, prices and POPs, and what every POP did this turn.

- `dev-ui <folder>` opens the window with a setting loaded.
- `dev-ui` alone opens it with an **Open setting...** button (a system folder picker; cancelling changes nothing).

For example: `cargo run -p policyspace-dev-ui -- settings/sample`.

#### Reading the map

Each hex is a pointy-top hexagon, rows run east-west, `q` increases eastward and `r` increases southward, exactly as in [The map](#The-map). The sample setting shows as a hexagon of 19 hexes. When hexes are drawn as hexagons, a hex with at least one deposit shows a small filled circle below its label and a hex with at least one POP a small hollow circle above it, so you can see where to click. A hex can have both marks, either, or neither; the sample's center has both.

**Controls**

- Drag to pan; scroll to zoom toward the cursor; **Fit** shows the whole map again.
- When zoomed in far enough, each hex shows its coordinates as `q, r` (the center of the sample is `0, 0`).
- Hovering highlights a hex and shows its coordinates in the status bar.
- The status bar also shows the setting name, the hex count and the current turn.

Unusual maps are handled: an empty map shows "This setting has no hexes."; holes, islands and negative coordinates draw correctly; for maps spanning huge distances, hexes too small to see are drawn as dots (with no marks), and zooming in turns them back into hexagons.

#### Inspecting a hex

Click a hex to select it (a thick accent-colored outline, distinct from the hover highlight). When hexes are drawn as dots, a click near a dot selects it. Click empty space or press Esc to clear. Dragging to pan never changes the selection.

The **hex panel** on the right shows, for the selected hex:

- its coordinates;
- its deposits grouped by type, for example `arable_land x 3: 4 unimproved_grain each, 12 total`;
- its stockpile of unowned goods: each good's total, and the parts of it with a countdown to when they perish: `perishes this turn`, `perishes in N turns`, or `never perishes`;
- its **market**: the hex's currency (`Currency: silver`), then each posted price as the amount of currency one unit costs (`grain: 2 silver`), in the order the goods are defined. A hex with a market but no prices says `No prices posted.`, and a hex with no market says `No market.`;
- its **POPs**: one collapsible section per POP, open at first, headed by its type and current size (`peasant × 61.6`), or `No POPs.`

Reading the countdown: suppose hex `(0, 0)` holds 15 of a good that lasts 3 turns, at turn 2, with 5 produced each turn. The panel lists 5 that perish this turn, 5 that perish in 1 turn and 5 that perish in 2 turns: the oldest batch is about to go, the newest has two more turns to live.

**What "this turn" means.** Between steps the world has just been endowed, traded, consumed and grown, and nothing has perished yet (see [Running a setting](#Running-a-setting)). So the panel shows the outcome of the turn in the status bar: each POP's size after that turn's growth and its stockpile after consumption. Each step replaces it all with the next turn's figures. A POP section you have collapsed stays collapsed when you step.

**A POP section**, for the sample's center peasant at turn 0, looks like this:

```
peasant × 61.6
  Started at 60; index 0.6667; rate +2.67%
  Income 600 silver; subsistence cost 360 silver; supernumerary 240 silver

  Good              Held   Demanded   Subsistence   Sold   Bought   Consumed
  unskilled_labor    600        168           120    432        0        168
  grain                0        216           120      0      216        216

  Carried over: 0 silver
  Stockpile: Empty.
```

- The **heading** is the type and the size after this turn's growth.
- The **growth line** gives the size the turn began with, the subsistence index and the growth rate (see [How POPs grow](#How-POPs-grow)). 60 peasants with 240 silver to spare against a subsistence cost of 360 have an index of 240 / 360 = 0.6667, a rate of +2.67%, and so become 61.6. Hovering the line shows the index and rate unrounded.
- The **demand line** is the POP's income, what its subsistence bundle costs at the hex's prices, and the difference, all in the hex's currency: 600 labor at 1 silver, against 120 labor at 1 and 120 grain at 2.
- The **goods table** has one row per good involved: what the POP held when it came to market, what it wanted to end up with, what it needs to subsist (its type's amount times its starting size), and what it sold, bought and consumed. Here the 240 spare silver went 0.2 to labor and 0.8 to grain, so the POP kept 168 labor and bought 216 grain. **A row in red means the POP wanted less of that good than it needs to subsist**, so a struggling POP and the good it lacks stand out.
- **Carried over** is the currency left after consumption: usually 0 or a tiny rounding sliver.
- The **stockpile** is the POP's own, written exactly like the hex's.

The north-east peasant on `(2, -2)`, whose hex shows `No market.`, makes the contrast: its section reads `peasant × 10.8`, `Started at 12; index -1.0000; rate -10.00%`, an income of 0, and its grain row in red (demanded 0 against a subsistence need of 24), while all 120 of its labor is held, demanded and consumed as leisure.

Sizes and amounts are written in full, with long decimals after a few turns, as in the turn log. Only the index (4 decimals), the rate (a percentage to 2 decimals) and the utility shares (up to 4 decimals) are rounded, and hovering shows the full values.

Other messages: "Click a hex to inspect it.", "No deposits.", "Empty.", "No market.", "No prices posted.", "No POPs." and "This setting has no hexes."

#### Stepping turns

The **Step** and **Step x10** buttons in the top bar advance the world one or ten turns (shortcuts: Space and Shift+Space). They work whenever the setting loaded without errors, even one with no hexes. The status bar's turn follows. The world starts at turn 0 on every load.

#### The world panel

The **world panel** on the left has four sections, which collapse; the panels can be resized.

- **Turn log:** every turn report since the load, turn 0 included, newest first, in the same wording as `check --turns` (see [Turn reports](#Turn-reports)).
- **Goods:** each good's name, how long it lasts, whether it is movable or fixed, and two amounts: **In hexes**, the unowned deposit output held in hex stockpiles, and **In POPs**, what the POPs own. In the sample the raw goods appear only under In hexes and silver only under In POPs.
- **Resource types:** each type's name, the good it produces, its output per deposit and its deposit count.
- **POP types:** each type's name, its endowment (or `nothing`), its utility goods with their subsistence amounts and shares as the engine uses them (hover for the full value), its growth parameters, the number of POPs of the type and their total size. The total sizes add up to the turn log's `population`.

#### The author's loop

1. Edit a setting file (for example, change some prices, deposits or POPs).
2. Click **Reload**. The world returns to turn 0, with a log of one line. Your pan and zoom stay, and so does your selection if the reloaded map still has a hex at those coordinates.
3. Step a few turns and check the stockpiles, prices and POPs.

If the setting has errors, the window shows the full error list (the same lines the checker prints) and no map or side panels, and the Step buttons are disabled. Fix the files and Reload again. A failed Reload doesn't lose the selection. A broken folder given on the command line opens the window showing its errors. Opening a different folder clears the selection and the log.

#### Not available yet

Auto-run, stepping by another number, coloring hexes by data (population included), a POP's history over turns or charts of it, editing the world, and selecting several hexes.

More detail: [docs/dev-ui.md](docs/dev-ui.md) for the panels and marks, with [docs/world.md](docs/world.md) for turns and stockpiles, [docs/market.md](docs/market.md) and [docs/currency.md](docs/currency.md) for the Market section, and [docs/demand.md](docs/demand.md), [docs/trade.md](docs/trade.md) and [docs/growth.md](docs/growth.md) for the numbers in a POP section.

## Technical documentation

For developers who want to understand or modify the project, see the [technical documentation](docs/README.md).